# Intel® Extension for Scikit-learn Random Forest for Iris dataset

In [138]:
import numpy as np
import time
from sklearn import metrics
from sklearn.datasets import fetch_openml
from IPython.display import HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Download the data from OpenML

In [ ]:
# Load Iris dataset using fetch_openml from sklearn
iris = fetch_openml(name='iris', version=1, as_frame=False)
X, y = iris.data, iris.target

# Encode target variable
le = LabelEncoder()
y = le.fit_transform(y)

Split the data into train and test sets

In [141]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Patch original Scikit-learn with Intel® Extension for Scikit-learn
Intel® Extension for Scikit-learn (previously known as daal4py) contains drop-in replacement functionality for the stock Scikit-learn package. You can take advantage of the performance optimizations of Intel® Extension for Scikit-learn by adding just two lines of code before the usual Scikit-learn imports:

In [142]:
from sklearnex import patch_sklearn

patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


Intel® Extension for Scikit-learn patching affects performance of specific Scikit-learn functionality. Refer to the [list of supported algorithms and parameters](https://uxlfoundation.github.io/scikit-learn-intelex/latest/algorithms.html) for details. In cases when unsupported parameters are used, the package fallbacks into original Scikit-learn. If the patching does not cover your scenarios, [submit an issue on GitHub](https://github.com/uxlfoundation/scikit-learn-intelex/issues).

Training Random Forest algorithm with Intel® Extension for Scikit-learn for Iris dataset


In [143]:
start_time = time.time()
intel_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42
)
intel_rf.fit(X_train, y_train)
intel_predictions = intel_rf.predict(X_test)
train_patched = time.time() - start_time
intel_accuracy = accuracy_score(y_test, intel_predictions)

f"Intel® extension for Scikit-learn time: {train_patched:.2f} s"


'Intel® extension for Scikit-learn time: 0.30 s'

Predict and get a result of the RandomForestClassifier algorithm with Intel® Extension for Scikit-learn

In [144]:
print("\nIntel Extension Results:")
print("Accuracy:", intel_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, intel_predictions))
print(f"Training Time: {train_patched:.4f} seconds")



Intel Extension Results:
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

Training Time: 0.3006 seconds


### Train the same algorithm with original Scikit-learn
In order to cancel optimizations, we use *unpatch_sklearn* and reimport the class RandomForestClassifier.

In [145]:
from sklearnex import unpatch_sklearn

unpatch_sklearn()

Training Random Forest algorithm with original Scikit-learn library for Iris dataset


In [146]:
start_time = time.time()
sk_rf = RandomForestClassifier(
    n_estimators=200,  # Increased number of trees
    max_depth=20,      # Added depth to increase complexity
    random_state=42
)
sk_rf.fit(X_train, y_train)
sk_predictions = sk_rf.predict(X_test)
train_unpatched = time.time() - start_time
original_accuracy = accuracy_score(y_test, sk_predictions)

f"Original Scikit-learn time: {train_unpatched:.2f} s"


'Original Scikit-learn time: 0.46 s'

In [147]:
print("\nStandard Scikit-learn Results:")
print("Accuracy:", original_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, sk_predictions))
print(f"Training Time: {train_unpatched:.4f} seconds")


Standard Scikit-learn Results:
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

Training Time: 0.4641 seconds


In [148]:
HTML(
    f"<h3>Compare MSE metric of patched Scikit-learn and original</h3>"
    f"Accuracy of patched Scikit-learn: {intel_accuracy} <br>"
    f"Accuracy of unpatched Scikit-learn: {original_accuracy} <br>"
    f"Accuracy ratio: {intel_accuracy/original_accuracy} <br>"
    f"<h3>With Scikit-learn-intelex patching you can:</h3>"
    f"<ul>"
    f"<li>Use your Scikit-learn code for training and prediction with minimal changes (a couple of lines of code);</li>"
    f"<li>Fast execution training and prediction of Scikit-learn models;</li>"
    f"<li>Get the similar quality</li>"
    f"<li>Get speedup in <strong>{(train_unpatched/train_patched):.1f}</strong> times.</li>"
    f"</ul>"
)